# Phase B.X - Classification en zero-shot et few-shot

Ce notebook intègre les approches fondées sur un modèle de langage

- **Classification en zero-shot et few-shot** :
À partir du texte d’une revue, demander à un LLM de prédire directement si l’avis est
positif ou négatif, sans (ou avec très peu de) données d’entraînement.


## 0. Préparation des modèles

Le model choisie : Llama3.2

La collection Meta Llama 3.2 de grands modèles de langage multilingues (LLM) est un ensemble de modèles génératifs pré-entraînés et optimisés pour les instructions, disponibles en tailles 1 et 3 milliards (texte en entrée/texte en sortie). Les modèles Llama 3.2 optimisés pour le texte uniquement sont conçus pour les cas d'utilisation de dialogues multilingues, notamment la recherche et la synthèse automatiques.

In [1]:
import sys
from langchain_ollama import ChatOllama
from langchain_core.callbacks.manager import CallbackManager
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

def get_local_llm(model_name="llama3.2"):
    """
    Configure et retourne une instance du modèle local via Ollama.
    """
    print(f"Connexion au modèle local '{model_name}' via Ollama...")

    try:
        llm = ChatOllama(
            model=model_name,
            temperature=0,
            callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
        )
        
        return llm

    except Exception as e:
        print(f"\n❌ ERREUR CRITIQUE : Impossible de joindre Ollama.")
        print(f"Détails : {e}")
        print("-" * 40)
        return None

c:\Users\cleme\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Prendre des "reviews" pour tests

In [2]:
import json
import random

reviews_file = "../data/raw/yelp_academic_reviews4students.jsonl"

reviews = []
with open(reviews_file, 'r', encoding='utf-8') as f:
    for line in f:
        reviews.append(json.loads(line))

# On filtre pour exclure les notes de 3 (neutres)
filtered_reviews = [r for r in reviews if r['stars'] != 3]

# Prendre 10 reviews aléatoires parmi les filtrées
sample_reviews = random.sample(filtered_reviews, min(10, len(filtered_reviews)))


### Créations des labels

In [3]:
def get_true_label(stars):
    if stars > 3:
        return "Positif"
    else:
        return "Négatif"

---

In [4]:
import pandas as pd

results_data = []

llm = get_local_llm()

Connexion au modèle local 'llama3.2' via Ollama...


## 1. Zero-shot

In [5]:
from langchain_core.prompts import PromptTemplate

template_zero_structured = """
Tu es un expert en analyse de sentiments.
Tâche : Analyse l'avis suivant.
Format de réponse attendu :
CLASSIFICATION : [STRICTEMENT "Positif" ou "Négatif"]
RAISONNEMENT : [Une phrase courte expliquant pourquoi]

Avis : {texte_avis}
Réponse :
"""

prompt_zero = PromptTemplate.from_template(template_zero_structured)
chain_zero = prompt_zero | llm

print("Lancement de l'analyse Zero-shot...")

for i, review in enumerate(sample_reviews, 1):
    text = review['text']
    true_label = get_true_label(review['stars']) # On récupère le vrai label
    
    response = chain_zero.invoke({"texte_avis": text})
    content = response.content
    
    # Parsing (Extraction simple basée sur le format demandé)
    label = "Inconnu"
    reasoning = "Pas d'explication trouvée"
    
    try:
        # On découpe le texte pour trouver les clés
        lines = content.split('\n')
        for line in lines:
            if "CLASSIFICATION :" in line:
                label = line.split("CLASSIFICATION :")[1].strip()
            if "RAISONNEMENT :" in line:
                reasoning = line.split("RAISONNEMENT :")[1].strip()
    except Exception as e:
        print(f"Erreur de parsing sur l'avis {i}: {e}")

    results_data.append({
        "id_review": i,
        "text": text,
        "mode": "Zero-shot",
        "true_label": true_label,
        "prediction": label,
        "reasoning": reasoning
    })
    
    print(f"Avis {i} traité.")
    print(response.content)


Lancement de l'analyse Zero-shot...
Avis 1 traité.
CLASSIFICATION : Positif
RAISONNEMENT : L'avis contient des mots et expressions positives tels que "amazing", "serene", "clean", "friendly", "top notch", "great bonus" et "THE BEST", qui démontrent une satisfaction totale de la part du client.
Avis 2 traité.
CLASSIFICATION : Négatif
RAISONNEMENT : L'auteur de l'avis exprime une forte déception et une insatisfaction avec son expérience à PJ's, notamment en raison du manque de qualité de sa commande (burger et frites non chaudes) et la mauvaise cuisson du burger.
Avis 3 traité.
CLASSIFICATION : Positif
RAISONNEMENT : L'auteur apprécie les prix de l'heure heureuse et la qualité du décor, ainsi que le fait qu'il s'agit d'un bon rapport qualité-prix avec des boissons et du sushi à 20 dollars.
Avis 4 traité.
CLASSIFICATION : Positif
RAISONNEMENT : L'auteur exprime des sentiments positifs envers Legal Sea Foods, mentionnant la qualité de la nourriture, le service exceptionnel et l'atmosphère 

## 2. Few-shot

In [6]:
template_few_structured = """
Tu es un expert en analyse de sentiments.
Tâche : Analyse l'avis suivant.
Format de réponse attendu :
CLASSIFICATION : [STRICTEMENT "Positif" ou "Négatif"]
RAISONNEMENT : [Une phrase courte expliquant pourquoi]

Exemple 1 :
Avis : "Went for lunch and found that my burger was meh.  What was obvious was that the focus of the burgers is the amount of different and random crap they can pile on it and not the flavor of the meat.  My burger patty seemed steamed and appeared to be a preformed patty, contrary to what is stated on the menu.    I can get ground beef from Kroger and make a burger that blows them out of the water."
Réponse : Négatif

Exemple 2 :
Avis : "I needed a new tires for my wife's car. They had to special order it and had it the next day, I dropped it off in the morning before work and they called a few hours later and the car was ready. It was quick and efficient, and the woman who helped me was awesome."
Réponse : Positif

Exemple 3 :
Avis : "Jim Woltman who works at Goleta Honda is 5 stars!! He is knowledgeable, helpful, and so personable.  He did a fantastic job on my Honda.  Thank you Jim!!!  And thank you Honda for having such a fabulous employee!"
Réponse : Positif

Tâche : Classifie l'avis suivant en utilisant la même logique.
Avis : {texte_avis}
Réponse :
"""

prompt_few = PromptTemplate.from_template(template_few_structured)
chain_few = prompt_few | llm

# 2. Boucle Few-shot
print("Lancement de l'analyse Few-shot...")

for i, review in enumerate(sample_reviews, 1):
    text = review['text']
    true_label = get_true_label(review['stars'])
    
    response = chain_few.invoke({"texte_avis": text})
    content = response.content
    
    # Parsing similaire
    label = "Inconnu"
    reasoning = "Pas d'explication trouvée"
    
    try:
        lines = content.split('\n')
        for line in lines:
            if "CLASSIFICATION :" in line:
                label = line.split("CLASSIFICATION :")[1].strip()
            if "RAISONNEMENT :" in line:
                reasoning = line.split("RAISONNEMENT :")[1].strip()
    except:
        pass

    results_data.append({
        "id_review": i,
        "text": text,
        "mode": "Few-shot",
        "true_label": true_label,
        "prediction": label,
        "reasoning": reasoning
    })
    
    print(f"Avis {i} (Few-shot) traité.")
    print(response.content)

Lancement de l'analyse Few-shot...
Avis 1 (Few-shot) traité.
CLASSIFICATION : Positif
RAISONNEMENT : L'auteur utilise des mots positifs tels que "amazing", "serene", "clean", "friendly" et exprime sa satisfaction avec la qualité du service, les options disponibles, les prix et les avantages offerts (vins gratuits, massages, etc.).
Avis 2 (Few-shot) traité.
CLASSIFICATION : Négatif
RAISONNEMENT : L'auteur de l'avis est profondément déçu et exprime sa mécontentement envers la qualité du repas, notamment la température insuffisante des aliments et le manque de décence de la part du manager.
Avis 3 (Few-shot) traité.
CLASSIFICATION : Positif
RAISONNEMENT : Bien que le client mentionne qu'il n'est pas impressionné par la qualité du sushi, il souligne les avantages de l'heure happy hour et de la qualité des boissons, ce qui fait prédominer son avis positif.
Avis 4 (Few-shot) traité.
CLASSIFICATION : Positif
RAISONNEMENT : L'auteur exprime une satisfaction générale pour la qualité des produit

## 3. Comparatif

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Préparation des données
df_results = pd.DataFrame(results_data)

# Nettoyage : On s'assure que les labels sont comparables (enlever espaces/casse)
df_results['prediction'] = df_results['prediction'].str.strip().str.capitalize()
df_results['true_label'] = df_results['true_label'].str.strip().str.capitalize()

# Filtrer les labels "Inconnu" et les prédictions invalides
df_results_clean = df_results[
    (df_results['prediction'] != 'Inconnu') & 
    (df_results['prediction'].isin(['Positif', 'Négatif']))
].copy()

# Liste des modes à évaluer
modes = df_results_clean['mode'].unique()

print("📊 ANALYSE DES PERFORMANCES")
print("="*50)

summary_metrics = []

for mode in modes:
    df_mode = df_results_clean[df_results_clean['mode'] == mode]
    
    if len(df_mode) == 0:
        print(f"⚠️  Pas de données valides pour le mode {mode}")
        continue
    
    # Calcul des métriques
    acc = accuracy_score(df_mode['true_label'], df_mode['prediction'])
    report = classification_report(df_mode['true_label'], df_mode['prediction'], output_dict=True)
    
    summary_metrics.append({
        "Méthode": mode,
        "Accuracy": f"{acc:.2%}",
        "F1-Score (Positif)": f"{report.get('Positif', {}).get('f1-score', 0):.2f}",
        "F1-Score (Négatif)": f"{report.get('Négatif', {}).get('f1-score', 0):.2f}"
    })

    print(f"\n▶ Rapport détaillé pour le mode : {mode}")
    print("-" * 40)
    print(classification_report(df_mode['true_label'], df_mode['prediction']))

ℹ️  0 lignes filtrées (labels invalides)
✓ 20 lignes valides pour l'analyse

📊 ANALYSE DES PERFORMANCES

▶ Rapport détaillé pour le mode : Zero-shot
----------------------------------------
              precision    recall  f1-score   support

     Négatif       0.67      1.00      0.80         2
     Positif       1.00      0.88      0.93         8

    accuracy                           0.90        10
   macro avg       0.83      0.94      0.87        10
weighted avg       0.93      0.90      0.91        10


▶ Rapport détaillé pour le mode : Few-shot
----------------------------------------
              precision    recall  f1-score   support

     Négatif       0.67      1.00      0.80         2
     Positif       1.00      0.88      0.93         8

    accuracy                           0.90        10
   macro avg       0.83      0.94      0.87        10
weighted avg       0.93      0.90      0.91        10



,Méthode,Accuracy,F1-Score (Positif),F1-Score (Négatif)
0,Zero-shot,90.00%,0.93,0.80
1,Few-shot,90.00%,0.93,0.80
